# G20: on-policy distillation data stage (student E4B rollouts on the 171-task pool + 31B teacher scoring)

Stage 1: deployed E4B arm with capture, 6 chunks. Stage 2: 31B scores every main-agent leaf with prompt_logprobs (top-8). Output: `g20/capture.jsonl.gz`, `g20/teacher_scores.npz`, `g20/teacher_meta.json`, `g20/e1_*/`, `g20/g20_summary.json`.

In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

## Setup

In [ ]:
import json, os, random, shutil, sys, threading, time, traceback
from pathlib import Path
WORKING_DIR = Path('/kaggle/working'); WORKING_DIR.mkdir(parents=True, exist_ok=True)
G2 = WORKING_DIR / 'g20'; G2.mkdir(exist_ok=True); CODE = G2 / 'code'; CODE.mkdir(exist_ok=True)
RESULTS = {'started': time.strftime('%Y-%m-%d %H:%M:%S')}
import torch
_gpus = [(torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())]
print('GPUs:', _gpus)
assert len(_gpus) == 4 and all(c[0] >= 8 for _, c in _gpus), f'expected 4x L4 (compute capability >= 8), got {_gpus}'
def save_results(): (G2 / 'g20_summary.json').write_text(json.dumps(RESULTS, indent=1, default=str))
BUDGET_MIN = 195; T_START = time.time()
def _watchdog():   # a hung round must not burn GPU quota
    RESULTS['watchdog'] = f'fired after {BUDGET_MIN} min'; save_results(); os._exit(3)
_wd = threading.Timer(BUDGET_MIN * 60, _watchdog); _wd.daemon = True; _wd.start()
for n, txt in {"capture_proxy.py": "\"\"\"Transparent logging reverse proxy between the harness (LiteLLM -> ADK) and vLLM, for G2 data collection.\n\nFor every non-streaming POST .../chat/completions it records, gzip-compressed, one JSON line with the FULL request body (messages incl. reasoning,\ntools, chat_template_kwargs, sampling params) and the FULL response body. It asks vLLM for `return_token_ids` so the record also carries the exact\nprompt token ids and completion token ids (no re-rendering guesses); if the server rejects that parameter it retries without it and keeps going.\nThe harness receives the response with the token-id fields stripped, i.e. exactly what it would get without the proxy.\nEverything else (GET /v1/models, /health, streaming requests) is passed through untouched.\n\"\"\"\n\nimport gzip\nimport hashlib\nimport json\nimport threading\nimport time\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\n\nimport requests\n\n\ndef task_key(messages: list[dict]) -> str:\n    \"\"\"Stable id of the agent session a request belongs to: hash of the system prompt + the first user message (the task statement).\"\"\"\n    sys_txt = next((m.get(\"content\") for m in messages if m.get(\"role\") == \"system\"), \"\") or \"\"\n    usr_txt = next((m.get(\"content\") for m in messages if m.get(\"role\") == \"user\"), \"\") or \"\"\n    h = hashlib.sha1()\n    for part in (sys_txt, usr_txt):\n        h.update((part if isinstance(part, str) else json.dumps(part, sort_keys=True)).encode(\"utf-8\", \"ignore\"))\n        h.update(b\"\\x00\")\n    return h.hexdigest()[:16]\n\n\nclass CaptureProxy:\n    def __init__(self, upstream: str, out_path: str, port: int = 8001, host: str = \"127.0.0.1\", ask_token_ids: bool = True):\n        self.upstream = upstream.rstrip(\"/\")          # e.g. http://127.0.0.1:8000\n        self.out_path = out_path\n        self.port, self.host = port, host\n        self.ask_token_ids = ask_token_ids\n        self.round_tag = \"r0\"\n        self.tokenize_fallback = False        # if True and vLLM returns no token ids, ask /tokenize for the exact prompt ids instead\n        self.n_records = 0\n        self.n_with_ids = 0\n        self.n_errors = 0\n        self._lock = threading.Lock()\n        self._httpd: ThreadingHTTPServer | None = None\n\n    # ---------------------------------------------------------------- lifecycle\n    def start(self) -> \"CaptureProxy\":\n        proxy = self\n\n        class H(BaseHTTPRequestHandler):\n            protocol_version = \"HTTP/1.1\"\n\n            def log_message(self, *a):  # silence\n                pass\n\n            def _relay(self, status: int, body: bytes, ctype: str = \"application/json\"):\n                self.send_response(status)\n                self.send_header(\"Content-Type\", ctype)\n                self.send_header(\"Content-Length\", str(len(body)))\n                self.end_headers()\n                self.wfile.write(body)\n\n            def do_GET(self):\n                try:\n                    r = requests.get(proxy.upstream + self.path, timeout=60)\n                    self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n                except Exception as e:  # noqa: BLE001\n                    self._relay(502, json.dumps({\"error\": str(e)}).encode())\n\n            def do_POST(self):\n                raw = self.rfile.read(int(self.headers.get(\"Content-Length\", \"0\")))\n                try:\n                    req = json.loads(raw) if raw else {}\n                except Exception:  # noqa: BLE001\n                    req = None\n                is_chat = self.path.rstrip(\"/\").endswith(\"/chat/completions\") and isinstance(req, dict) and not req.get(\"stream\")\n                if not is_chat:                      # pass-through (streaming, other endpoints)\n                    try:\n                        r = requests.post(proxy.upstream + self.path, data=raw, headers={\"Content-Type\": \"application/json\"}, timeout=1800)\n                        self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n                    except Exception as e:  # noqa: BLE001\n                        self._relay(502, json.dumps({\"error\": str(e)}).encode())\n                    return\n                t0 = time.time()\n                asked = proxy.ask_token_ids           # decided ONCE per request (another thread may flip the flag meanwhile)\n                body = dict(req, return_token_ids=True) if asked else req\n                try:\n                    r = requests.post(proxy.upstream + self.path, json=body, timeout=1800)\n                    if asked and r.status_code == 400 and \"return_token_ids\" in r.text:\n                        proxy.ask_token_ids = False   # this vLLM rejects the parameter: stop asking, redo this call plainly\n                        r = requests.post(proxy.upstream + self.path, json=req, timeout=1800)\n                except Exception as e:  # noqa: BLE001\n                    proxy.n_errors += 1\n                    self._relay(502, json.dumps({\"error\": f\"proxy upstream failure: {e}\"}).encode())\n                    return\n                try:\n                    resp = r.json()\n                except Exception:  # noqa: BLE001\n                    resp = None\n                if r.status_code == 200 and isinstance(resp, dict):\n                    proxy._record(req, resp, t0, time.time())\n                    clean = {k: v for k, v in resp.items() if k != \"prompt_token_ids\"}\n                    clean[\"choices\"] = [{k: v for k, v in ch.items() if k != \"token_ids\"} for ch in resp.get(\"choices\", [])]\n                    self._relay(200, json.dumps(clean).encode())\n                else:\n                    proxy.n_errors += 1\n                    self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n\n        self._httpd = ThreadingHTTPServer((self.host, self.port), H)\n        self._httpd.daemon_threads = True\n        threading.Thread(target=self._httpd.serve_forever, daemon=True).start()\n        return self\n\n    def stop(self) -> None:\n        if self._httpd:\n            self._httpd.shutdown()\n            self._httpd.server_close()\n\n    # ---------------------------------------------------------------- recording\n    def _record(self, req: dict, resp: dict, t0: float, t1: float) -> None:\n        has_ids = bool(resp.get(\"prompt_token_ids\")) and bool((resp.get(\"choices\") or [{}])[0].get(\"token_ids\"))\n        rec = {\"round\": self.round_tag, \"task_key\": task_key(req.get(\"messages\", [])), \"t0\": round(t0, 3), \"t1\": round(t1, 3), \"request\": req, \"response\": resp}\n        if not has_ids and self.tokenize_fallback:\n            try:\n                tk = {k: req[k] for k in (\"model\", \"messages\", \"tools\", \"chat_template_kwargs\") if k in req}\n                tk[\"add_generation_prompt\"] = True\n                r = requests.post(self.upstream + \"/tokenize\", json=tk, timeout=300)\n                if r.status_code == 200:\n                    rec[\"prompt_token_ids_tokenize\"] = r.json().get(\"tokens\")\n            except Exception:  # noqa: BLE001\n                pass\n        line = (json.dumps(rec, ensure_ascii=False) + \"\\n\").encode(\"utf-8\")\n        with self._lock:                             # one gzip member per record; the concatenation is a valid gzip stream\n            with open(self.out_path, \"ab\") as f:\n                f.write(gzip.compress(line, compresslevel=3))\n            self.n_records += 1\n            self.n_with_ids += has_ids\n\n    def stats(self) -> dict:\n        return {\"records\": self.n_records, \"with_token_ids\": self.n_with_ids, \"errors\": self.n_errors, \"asking_token_ids\": self.ask_token_ids, \"round\": self.round_tag}\n\n\ndef read_records(path: str):\n    \"\"\"Iterate the records of a capture file.\"\"\"\n    with gzip.open(path, \"rt\", encoding=\"utf-8\") as f:\n        for line in f:\n            if line.strip():\n                yield json.loads(line)\n"}.items(): (CODE / n).write_text(txt, encoding='utf-8')
(CODE / 'opd_score.py').write_text("\"\"\"G20 (on-policy distillation, data stage): teacher scoring of student contexts.\n\nThe student (E4B) runs the real harness and the capture proxy records every request with exact prompt and completion token ids. For each context segment of the main\nagent (one per capture task_key; compaction starts a new one) the LEAF is the request with the longest prompt, and its sequence is prompt ids + completion ids: it\ncontains every model turn of that segment (as re-rendered in history, the same form the SFT sequences use). The teacher (31B, same tokenizer and chat template,\nverified 60/60 on real captures) scores each leaf in one prefill through vLLM's /v1/completions with `prompt_logprobs`: per position the log-probability of the\nactual token and the teacher's top-k tokens. Loss spans (the student's own tokens) are derived offline from the capture, as in scripts/g10/build_sft_general.py.\n\nOutput (npz, rewritten as it grows): ids int32 [N], leaf_offsets int64 [L+1], n_prompt int32 [L], act_lp float32 [N] (teacher log p of the actual token; NaN at\nposition 0), top_ids int32 [N,k], top_lp float16 [N,k] (-inf padding), plus meta.json (task_key, round, t0 of the leaf request, seconds).\n\"\"\"\n\nimport gzip\nimport json\nimport math\nimport time\n\nimport numpy as np\nimport requests\n\nMAIN = \"You are an expert autonomous software engineer\"\n\n\ndef _text(c):\n    if isinstance(c, list):\n        return \"\".join(p.get(\"text\", \"\") for p in c if isinstance(p, dict))\n    return c or \"\"\n\n\ndef read_records(path):\n    with gzip.open(path, \"rt\", encoding=\"utf-8\") as f:\n        for line in f:\n            if line.strip():\n                yield json.loads(line)\n\n\ndef leaves(capture_path, round_prefix):\n    \"\"\"Main-agent leaves of the rounds whose tag starts with round_prefix, shortest first.\"\"\"\n    best = {}\n    for rec in read_records(capture_path):\n        if not str(rec.get(\"round\", \"\")).startswith(round_prefix):\n            continue\n        msgs = rec[\"request\"].get(\"messages\", [])\n        sysm = _text(next((m.get(\"content\") for m in msgs if m.get(\"role\") == \"system\"), \"\"))\n        if not sysm.startswith(MAIN):\n            continue\n        resp = rec.get(\"response\") or {}\n        p = resp.get(\"prompt_token_ids\")\n        c = ((resp.get(\"choices\") or [{}])[0]).get(\"token_ids\")\n        if not p or not c:\n            continue\n        k = rec[\"task_key\"]\n        if k not in best or len(p) > best[k][\"n_prompt\"]:\n            best[k] = {\"task_key\": k, \"round\": rec[\"round\"], \"t0\": rec.get(\"t0\"), \"n_prompt\": len(p), \"ids\": list(p) + list(c)}\n    return sorted(best.values(), key=lambda x: len(x[\"ids\"]))\n\n\ndef _tid(key):\n    s = str(key)\n    if s.startswith(\"token_id:\"):\n        s = s[len(\"token_id:\"):]\n    return int(s)\n\n\ndef _lp(v):\n    return float(v[\"logprob\"]) if isinstance(v, dict) else float(v)\n\n\ndef parse_prompt_logprobs(plp, ids, k):\n    \"\"\"vLLM prompt_logprobs (list, None at position 0, else {token_id: {logprob, rank, decoded_token}} with the actual token plus the top-k) -> arrays.\"\"\"\n    n = len(ids)\n    if len(plp) != n:\n        raise ValueError(f\"prompt_logprobs has {len(plp)} positions, sequence has {n}\")\n    act = np.full(n, np.nan, np.float32)\n    top_ids = np.full((n, k), -1, np.int32)\n    top_lp = np.full((n, k), -np.inf, np.float16)\n    for i, d in enumerate(plp):\n        if not d:\n            continue\n        items = sorted(((_tid(t), _lp(v)) for t, v in d.items()), key=lambda x: -x[1])\n        dd = dict(items)\n        if ids[i] not in dd:\n            raise ValueError(f\"actual token {ids[i]} missing at position {i}\")\n        act[i] = dd[ids[i]]\n        for j, (t, lp) in enumerate(items[:k]):\n            top_ids[i, j] = t\n            top_lp[i, j] = lp\n    return act, top_ids, top_lp\n\n\ndef score_leaf(url, model, ids, k, timeout=1800):\n    r = requests.post(url.rstrip(\"/\") + \"/v1/completions\",\n                      json={\"model\": model, \"prompt\": ids, \"max_tokens\": 1, \"temperature\": 0.0, \"prompt_logprobs\": k}, timeout=timeout)\n    if r.status_code != 200:\n        raise RuntimeError(f\"HTTP {r.status_code}: {r.text[:500]}\")\n    return r.json()[\"choices\"][0][\"prompt_logprobs\"]\n\n\nclass Store:\n    def __init__(self, k):\n        self.k, self.rows, self.meta = k, [], []\n\n    def add(self, leaf, act, top_ids, top_lp, seconds):\n        self.rows.append((np.asarray(leaf[\"ids\"], np.int32), leaf[\"n_prompt\"], act, top_ids, top_lp))\n        self.meta.append({\"task_key\": leaf[\"task_key\"], \"round\": leaf[\"round\"], \"t0\": leaf[\"t0\"], \"n_tokens\": len(leaf[\"ids\"]), \"n_prompt\": leaf[\"n_prompt\"], \"seconds\": round(seconds, 2)})\n\n    def save(self, npz_path, meta_path):\n        if not self.rows:\n            return\n        lens = [len(r[0]) for r in self.rows]\n        np.savez(npz_path, ids=np.concatenate([r[0] for r in self.rows]), leaf_offsets=np.concatenate([[0], np.cumsum(lens)]).astype(np.int64),\n                 n_prompt=np.array([r[1] for r in self.rows], np.int32), act_lp=np.concatenate([r[2] for r in self.rows]),\n                 top_ids=np.concatenate([r[3] for r in self.rows]), top_lp=np.concatenate([r[4] for r in self.rows]))\n        with open(meta_path, \"w\", encoding=\"utf-8\") as f:\n            json.dump({\"k\": self.k, \"leaves\": self.meta}, f)\n\n\ndef score_all(leaves_, job, store, out_of_time, on_progress, workers=2, every=10):\n    \"\"\"Run job(leaf) -> (act, top_ids, top_lp, seconds) over leaves with `workers` requests in flight; stop submitting once out_of_time() is True.\n    Failed leaves are counted and their first errors kept; on_progress() is called every `every` finished leaves and at the end. Returns (n_ok, errors).\"\"\"\n    from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait\n    it, pending, errors, n_done, n_ok = iter(leaves_), {}, [], 0, 0\n    with ThreadPoolExecutor(workers) as ex:\n        def fill():\n            while len(pending) < workers and not out_of_time():\n                leaf = next(it, None)\n                if leaf is None:\n                    return\n                pending[ex.submit(job, leaf)] = leaf\n        fill()\n        while pending:\n            done, _ = wait(list(pending), return_when=FIRST_COMPLETED)\n            for f in done:\n                leaf = pending.pop(f)\n                try:\n                    act, top_ids, top_lp, sec = f.result()\n                    store.add(leaf, act, top_ids, top_lp, sec)\n                    n_ok += 1\n                except Exception as e:  # noqa: BLE001\n                    errors.append(f\"{leaf['task_key']}: {str(e)[:300]}\")\n                n_done += 1\n                if n_done % every == 0:\n                    on_progress()\n            fill()\n    on_progress()\n    return n_ok, errors\n\n\ndef summary(store):\n    \"\"\"Mean teacher log p of the actual tokens over the completion part of each leaf (a rough 'how much does the teacher agree with the student' number).\"\"\"\n    vals = []\n    for ids, n_prompt, act, _, _ in store.rows:\n        a = act[n_prompt:]\n        a = a[~np.isnan(a)]\n        if len(a):\n            vals.append(float(a.mean()))\n    return {\"leaves\": len(store.rows), \"tokens\": int(sum(len(r[0]) for r in store.rows)),\n            \"mean_teacher_lp_last_completion\": round(sum(vals) / len(vals), 4) if vals else None,\n            \"frac_last_completion_tokens_teacher_lp_below_ln0.1\": round(float(np.mean(np.concatenate([r[2][r[1]:] for r in store.rows]) < math.log(0.1))), 4) if store.rows else None}\n", encoding='utf-8')
sys.path.insert(0, str(CODE))
# the competition is mounted at /kaggle/input/gemma-4-developer-agent/ (observed in G10 v2) with its own tasks.jsonl; the pool is the g9-build kernel output
_tj = [p for p in Path('/kaggle/input').rglob('tasks.jsonl') if 'gemma-4-g9-build' in p.parts and (p.parent / 'snapshots').is_dir()]
assert len(_tj) == 1, _tj
POOL = _tj[0].parent
DATA_DIR = POOL
assert (POOL / 'snapshots').is_dir() and (POOL / 'wheels').is_dir(), POOL
# side directory with the task dependencies (see scripts/g9/taskdeps.py): must exist on sys.path BEFORE any sandbox is created
exec("\"\"\"Unpack the shared wheel set into a side directory for the subprocess sandbox (used by the gold check and by the rollout notebooks).\n\nFacts established from the swegemma 0.2.11 sources and the pilots (2026-10-13):\n  * in subprocess mode swegemma does NOT inject wheels (install_test_dependencies returns early); the sandbox venv reaches the host through a `_host_env.pth` listing every\n    host sys.path entry that contains 'site-packages'; the venv also has include-system-site-packages, and entries added by that .pth come BEFORE the system dist-packages,\n    so anything in the side directory OVERRIDES the host's copy;\n  * therefore: never put pytest, its core dependencies or pytest PLUGINS in the side directory (a plugin such as pytest-sanic is auto-loaded for every task and crashes\n    importing a package that is not installed). The harness itself skips pytest, pytest_timeout, pytest_asyncio, pytest_httpbin, pytest_xdist, pluggy, pip, setuptools, wheel.\n\"\"\"\n\nimport re\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nDENY = {\"pytest\", \"pytest_timeout\", \"pytest_asyncio\", \"pytest_httpbin\", \"pytest_xdist\", \"pluggy\", \"pip\", \"setuptools\", \"wheel\", \"iniconfig\", \"packaging\", \"exceptiongroup\", \"tomli\"}\nPLUGIN_WHITELIST = {\"pytest_mock\", \"pytest_cov\", \"pytest_asyncio\", \"pytest_xdist\"}   # entry-point plugins that are inert unless the tests ask for them\n\n\ndef norm(name: str) -> str:\n    return re.sub(r\"[-_.]+\", \"_\", name).lower()\n\n\ndef classify(wheel: Path) -> str:\n    \"\"\"'ok' | 'deny' | 'plugin'. A pytest PLUGIN package (name starts with pytest_ / pytest-) is excluded unless whitelisted: its entry point is auto-loaded for every task.\n    Ordinary libraries that merely ship a pytest11 entry point (platformdirs, hypothesis, ...) are fine: they are installed anyway, so their plugin imports cleanly.\"\"\"\n    pkg = norm(wheel.name.split(\"-\", 1)[0])\n    if pkg in DENY:\n        return \"deny\"\n    if pkg.startswith(\"pytest_\") and pkg not in PLUGIN_WHITELIST:\n        return \"plugin\"\n    return \"ok\"\n\n\ndef strip_pytest_entry_points(target: Path) -> list:\n    \"\"\"Remove the [pytest11] section from every dist-info in the side directory (except whitelisted plugins). pytest auto-loads entry-point plugins for EVERY task, and a\n    plugin from an ordinary package can crash at import (pilot v3: hydra-core's plugin imported omegaconf/antlr4 with a mismatching runtime in all 12 tasks).\"\"\"\n    stripped = []\n    for ep in Path(target).glob(\"*.dist-info/entry_points.txt\"):\n        pkg = norm(ep.parent.name.split(\"-\", 1)[0])   # \"pytest_mock-3.14.0.dist-info\" -> \"pytest_mock\"\n        text = ep.read_text(encoding=\"utf-8\", errors=\"ignore\")\n        if \"[pytest11]\" not in text or pkg in PLUGIN_WHITELIST:\n            continue\n        out, skip = [], False\n        for line in text.splitlines():\n            if line.strip().startswith(\"[\"):\n                skip = line.strip() == \"[pytest11]\"\n            if not skip:\n                out.append(line)\n        ep.write_text(\"\\n\".join(out) + \"\\n\", encoding=\"utf-8\")\n        stripped.append(pkg)\n    return sorted(stripped)\n\n\ndef build_taskdeps(wheels_dir: Path, target: Path) -> dict:\n    wheels = sorted(Path(wheels_dir).glob(\"*.whl\"))\n    kinds = {w: classify(w) for w in wheels}\n    use = [str(w) for w, k in kinds.items() if k == \"ok\"]\n    target.mkdir(parents=True, exist_ok=True)\n    r = subprocess.run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"--no-index\", \"--no-deps\", \"--target\", str(target), *use], capture_output=True, text=True)\n    stripped = strip_pytest_entry_points(target) if r.returncode == 0 else []\n    return {\"rc\": r.returncode, \"installed\": len(use), \"pytest_entry_points_stripped\": stripped, \"denied\": sorted(w.name for w, k in kinds.items() if k == \"deny\"),\n            \"plugins_excluded\": sorted(w.name for w, k in kinds.items() if k == \"plugin\"), \"stderr_tail\": (r.stderr or \"\")[-300:]}\n")
_rep = build_taskdeps(POOL / 'wheels', Path('/tmp/taskdeps/site-packages'))
print('taskdeps: rc', _rep['rc'], '| installed', _rep['installed'], '| denied', len(_rep['denied']), '| pytest11 stripped', len(_rep['pytest_entry_points_stripped']))
assert _rep['rc'] == 0, _rep['stderr_tail']
sys.path.append('/tmp/taskdeps/site-packages')
(WORKING_DIR / 'no_graphs').mkdir(exist_ok=True); (WORKING_DIR / 'no_embeddings').mkdir(exist_ok=True)
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
arm_dirs = [p.parent for p in Path('/kaggle/input').rglob('agent.yaml') if 'swe_scaffold_v5_e4b_base' in p.read_text(encoding='utf-8')]
assert len(arm_dirs) == 1, f'expected exactly one E4B arm, found {arm_dirs}'
ARM = WORKING_DIR / 'arm'; shutil.rmtree(ARM, ignore_errors=True); shutil.copytree(arm_dirs[0], ARM)
ARMS_NS = {}; exec('"""Config-screen arms (G6): copies of the B1 arm whose only difference is configs/sampling.yaml.\n\nT1: shorter thinking budget and output cap (cut the long turns that eat the 5-minute clock)\nT2: T1 + Gemma\'s usual sampling (temperature 1.0, top_p 0.95, top_k 64)\n"""\n\nimport re\nimport shutil\nfrom pathlib import Path\n\nARMS = {\n    "t1": dict(temperature=0.2, top_p=0.95, top_k=None, max_output_tokens=4096, thinking_budget=2048),\n    "t2": dict(temperature=1.0, top_p=0.95, top_k=64, max_output_tokens=4096, thinking_budget=2048),\n    "t4": dict(temperature=1.0, top_p=0.95, top_k=64, max_output_tokens=8192, thinking_budget=4096),   # G13 probe: B1 settings except Gemma\'s usual sampling\n    "e4b": dict(temperature=0.2, top_p=0.95, top_k=None, max_output_tokens=8192, thinking_budget=4096),   # G14a: E4B baseline with the B1/B2 sampling\n    "e4b_b2": dict(temperature=0.2, top_p=0.95, top_k=None, max_output_tokens=8192, thinking_budget=4096),   # G14c: E4B, B2 prompt (budget-agnostic)\n    "e4b_v": dict(temperature=0.2, top_p=0.95, top_k=None, max_output_tokens=8192, thinking_budget=4096),   # G14c: E4B, B2 prompt + reproduce-then-verify schedule\n    "t3": dict(temperature=0.2, top_p=0.95, top_k=None, max_output_tokens=3072, thinking_budget=1024),   # G12 (31B): half of T1\'s thinking budget\n}\n\n\ndef sampling_yaml(temperature, top_p, top_k, max_output_tokens, thinking_budget) -> str:\n    lines = [f"temperature: {temperature}", f"top_p: {top_p}"]\n    if top_k is not None:\n        lines.append(f"top_k: {top_k}")\n    lines += [f"max_output_tokens: {max_output_tokens}", "thinking_config:", f"  thinking_budget: {thinking_budget}", "  include_thoughts: true", ""]\n    return "\\n".join(lines)\n\n\ndef make_arm(src: Path, dst: Path, **kw) -> Path:\n    shutil.rmtree(dst, ignore_errors=True)\n    shutil.copytree(src, dst)\n    p = dst / "configs" / "sampling.yaml"\n    old = p.read_text(encoding="utf-8")\n    assert re.search(r"^temperature:", old, re.M) and "thinking_budget" in old, "unexpected sampling.yaml layout"\n    p.write_text(sampling_yaml(**kw), encoding="utf-8", newline="\\n")\n    return dst\n\n\ndef request_matches(request: dict, expect: dict) -> list:\n    """Compare one captured vLLM request body with the arm\'s settings; returns the list of mismatches (empty = OK)."""\n    bad = []\n    if request.get("temperature") != expect["temperature"]:\n        bad.append(("temperature", request.get("temperature"), expect["temperature"]))\n    if request.get("top_p") != expect["top_p"]:\n        bad.append(("top_p", request.get("top_p"), expect["top_p"]))\n    if expect.get("top_k") is not None and request.get("top_k") != expect["top_k"]:\n        bad.append(("top_k", request.get("top_k"), expect["top_k"]))\n    mo = request.get("max_completion_tokens", request.get("max_tokens"))\n    if mo != expect["max_output_tokens"]:\n        bad.append(("max_output_tokens", mo, expect["max_output_tokens"]))\n    if request.get("thinking_token_budget") != expect["thinking_budget"]:\n        bad.append(("thinking_token_budget", request.get("thinking_token_budget"), expect["thinking_budget"]))\n    return bad\n', ARMS_NS)   # ARMS, sampling_yaml, request_matches
SAMPLING = ARMS_NS['ARMS']['e4b']   # the deployed E4B arm is used UNCHANGED (on-policy for the submission); the config gate checks its requests against this
_ev = (ARM / 'eval_config.yaml').read_text(); assert 'max_tool_calls: 28' in _ev and 'max_time_minutes: 8' in _ev, _ev
assert (ARM / 'agent.yaml').read_text().count('gemma-4-e4b-it-qat-w4a16-ct') == 1 and 'gemma-4-31b-it-qat-w4a16-ct' not in (ARM / 'agent.yaml').read_text()
print('sampling.yaml of the arm:', (ARM / 'configs' / 'sampling.yaml').read_text().replace(chr(10), ' | '))
for must in ('agent.yaml', 'prompts/system.md', 'prompts/analyzer.md', 'configs/sampling.yaml', 'sub_agents/code_analyzer.yaml', 'eval_config.yaml'):
    assert (ARM / must).is_file(), f'arm missing {must}'
print('arm:', arm_dirs[0], '|', (ARM / 'configs/sampling.yaml').read_text().replace(chr(10), ' ')[-120:])
from swegemma.models import load_tasks
ALL_IDS = json.loads(next(Path('/kaggle/input').rglob('rollout_pool.json')).read_text())
by_id = {t.instance_id: t for t in load_tasks(TASKS_PATH)}
import numpy as _np
for _t in by_id.values():
    _short = _t.repo.split('/')[-1]
    (WORKING_DIR / 'no_graphs' / f'{_short}_{_t.base_commit}.json').write_text(json.dumps({'directed': True, 'multigraph': False, 'graph': {}, 'nodes': [], 'links': [], 'note': 'stub graph for rollouts, no code graph exists for this task'}))
    _np.savez(WORKING_DIR / 'no_embeddings' / f'{_short}_{_t.base_commit}.npz', ids=_np.array(['stub'] * 8), vectors=_np.zeros((8, 256), dtype='float32'))
print('stub graphs/embeddings written for', len(by_id), 'tasks')
missing = [i for i in ALL_IDS if i not in by_id]
assert not missing, f'task ids missing from the dataset: {missing}'
SMOKE_IDS = random.Random(5).sample(ALL_IDS, 4)
print(len(ALL_IDS), 'tasks;', 'smoke ids:', SMOKE_IDS)

## vLLM (real 31B, no adapter) + capture proxy

In [ ]:
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-e4b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-e4b-it-qat-w4a16-ct/2')
assert MODEL_PATH.is_dir(), MODEL_PATH
first_arm = ARM
try:   # swegemma >= 0.2.11 / adk_submission >= 0.2.13 (wheelhouse of 2026-10-09)
    from adk_submission import discover_declared_models
    _declared = discover_declared_models(first_arm); assert len(_declared) == 1, f'expected exactly one declared model, got {_declared}'
    declared_model = next(iter(_declared))
except ImportError:   # older wheelhouse
    from swegemma.models.discovery import validate_single_declared_model
    declared_model = validate_single_declared_model(first_arm)
adapters = discover_adapters(str(first_arm), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
vllm_cfg = VllmConfig(model=str(MODEL_PATH), port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768, dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=False,
    tensor_parallel_size=tp_size, startup_timeout=60 * 12)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')
import capture_proxy as CP
VLLM_URL = f'http://127.0.0.1:{server_instance.config.port}'
proxy = CP.CaptureProxy(VLLM_URL, str(G2 / 'capture.jsonl.gz'), port=8001).start()
server_instance.config.port = 8001      # the model registry reads base_url at creation time: from now on the harness talks to the capture proxy
models = server_instance.create_model_registry(aliases=[declared_model, TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')
print('harness endpoint (proxy):', server_instance.base_url, '-> upstream', VLLM_URL)

## Smoke round, capture gate, full rounds

In [ ]:
import asyncio, concurrent.futures, requests, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_fn):
    try: loop = asyncio.get_running_loop()
    except RuntimeError: loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro_fn())).result()
    return asyncio.run(coro_fn())

def budget_of(arm_dir):
    cfg = yaml.safe_load((arm_dir / 'eval_config.yaml').read_text(encoding='utf-8')).get('evaluation', {})
    out = {}
    for k, cast in (('timeout_seconds', int), ('max_tool_calls', int), ('max_time_minutes', float)):
        if k in cfg: out[k] = cast(cfg[k])
    t = cfg.get('max_turns', cfg.get('max_llm_calls'))
    if t is not None: out['max_turns'] = int(t)
    return out

limits, gen_constraints = build_submission_limits()
print('per-task budget:', budget_of(ARM))

def run_round(tag, ids):
    proxy.round_tag = tag
    rd = G2 / tag
    cfg = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=rd, submission_dir=ARM, models=models, sandbox='subprocess',
        task_ids=list(ids), concurrency=4, display_mode='quiet', limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=str(WORKING_DIR / 'no_graphs'), embeddings_dir=str(WORKING_DIR / 'no_embeddings'), wheels_dir=DATA_DIR / 'wheels', verbose=False, **budget_of(ARM))
    t0 = time.perf_counter()
    try:
        run_sync(Evaluator(cfg).run)
    except Exception:
        traceback.print_exc(); RESULTS[tag] = {'error': 'run raised', 'trace': traceback.format_exc()[-1500:]}; save_results(); return None
    rows = [json.loads(l) for l in open(rd / 'task_results.jsonl', encoding='utf-8')]
    RESULTS[tag] = {'n': len(rows), 'resolved': sum(r['resolved'] for r in rows), 'patch_written': sum(1 for r in rows if r['agent_patch_size'] > 0),
                    'time_cap_hit': sum(1 for r in rows if 'session timeout' in (r['error'] or '')), 'wall_s': round(time.perf_counter() - t0, 1),
                    'mean_task_s': round(sum(r['duration_seconds'] for r in rows) / max(1, len(rows)), 1), 'resolved_ids': sorted(r['instance_id'] for r in rows if r['resolved']),
                    'proxy': proxy.stats()}
    save_results(); print(tag, {k: v for k, v in RESULTS[tag].items() if k != 'resolved_ids'}, flush=True)
    return rows

# ------------------------------------------------------------------ SMOKE: 4 tasks, then a fail-fast capture gate (saves quota if capture is broken)
rows = run_round('smoke', SMOKE_IDS)
st = proxy.stats(); print('capture after smoke:', st)
gate = {'rows': bool(rows) and len(rows) == len(SMOKE_IDS), 'records >= 12': st['records'] >= 12, 'no proxy errors': st['errors'] == 0}
if st['with_token_ids'] == 0 and st['records'] > 0:      # vLLM returned no ids: switch on the /tokenize fallback for the rest of the run
    proxy.tokenize_fallback = True; print('WARNING: no token ids from vLLM; /tokenize fallback enabled')
try:   # cross-check: ids returned by vLLM == ids from /tokenize for the same request (the exactness of the whole capture rests on this)
    first = next(CP.read_records(str(G2 / 'capture.jsonl.gz')))
    rq = first['request']
    tk = {k: rq[k] for k in ('model', 'messages', 'tools', 'chat_template_kwargs') if k in rq}; tk['add_generation_prompt'] = True
    tok = requests.post(VLLM_URL + '/tokenize', json=tk, timeout=300).json().get('tokens')
    got = first['response'].get('prompt_token_ids') or first.get('prompt_token_ids_tokenize')
    RESULTS['token_id_crosscheck'] = {'tokenize_len': len(tok or []), 'captured_len': len(got or []), 'identical': tok == got}
except Exception:
    RESULTS['token_id_crosscheck'] = {'error': traceback.format_exc()[-800:]}
print('cross-check:', RESULTS['token_id_crosscheck']); save_results()
MAIN_PFX = 'You are an expert autonomous software engineer'
_reqs = [r['request'] for r in CP.read_records(str(G2 / 'capture.jsonl.gz')) if r['round'] == 'smoke' and (next((m.get('content') for m in r['request']['messages'] if m.get('role') == 'system'), '') or '').startswith(MAIN_PFX)]
_bad = [m for r in _reqs for m in ARMS_NS['request_matches'](r, SAMPLING)]
gate['E4B arm sampling settings reach vLLM'] = bool(_reqs) and not _bad
RESULTS['config_gate'] = {'main_agent_requests': len(_reqs), 'mismatches': _bad[:5]}; print('config gate:', RESULTS['config_gate'])
RESULTS['smoke_gate'] = gate
if not all(gate.values()):
    raise RuntimeError(f'SMOKE GATE FAILED, stopping before the expensive rounds: {gate} {st}')
print('smoke gate passed:', gate)

# ------------------------------------------------------------------ G20 stage 1: the student over the 171-task pool, in chunks (no new chunk after ROUND_STOP_MIN)
ROUND_STOP_MIN = 125
ids = list(ALL_IDS); random.Random(120).shuffle(ids)
CHUNKS = [ids[i::6] for i in range(6)]
for k, ch in enumerate(CHUNKS):
    elapsed_min = (time.time() - T_START) / 60
    if elapsed_min > ROUND_STOP_MIN:
        print(f'not starting chunk {k}: elapsed {elapsed_min:.0f} min > {ROUND_STOP_MIN}'); break
    run_round(f'e1_{k}', ch)
_rs = [RESULTS[t] for t in RESULTS if t.startswith('e1_') and 'n' in RESULTS[t]]
RESULTS['e1_total'] = {'n': sum(r['n'] for r in _rs), 'resolved': sum(r['resolved'] for r in _rs), 'patch_written': sum(r['patch_written'] for r in _rs),
                       'time_cap_hit': sum(r['time_cap_hit'] for r in _rs), 'chunks': len(_rs), 'minutes': round((time.time() - T_START) / 60, 1)}
save_results(); print('student round:', RESULTS['e1_total'], '| capture:', proxy.stats(), flush=True)

# ------------------------------------------------------------------ G20 stage 2: the 31B teacher scores the student's leaves (prompt_logprobs)
import subprocess
import opd_score as O
proxy.stop()
try:
    server_instance.stop()
except Exception as e:
    print('server_instance.stop() failed:', e)
def gpu_used():
    out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.split()
    return [int(x) for x in out]
for _ in range(36):
    if max(gpu_used()) < 1500: break
    time.sleep(5)
else:
    subprocess.run(['pkill', '-f', 'vllm'], check=False); time.sleep(30)
RESULTS['gpu_mib_after_student'] = gpu_used(); print('GPU MiB after stopping the student:', RESULTS['gpu_mib_after_student'], flush=True)
LEAVES = O.leaves(str(G2 / 'capture.jsonl.gz'), 'e1_')
RESULTS['leaves'] = {'n': len(LEAVES), 'tokens': sum(len(l['ids']) for l in LEAVES), 'max_len': max((len(l['ids']) for l in LEAVES), default=0)}
save_results(); print('leaves:', RESULTS['leaves'], flush=True)
K = 8
T_MODEL = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2'); assert T_MODEL.is_dir(), T_MODEL
tcfg = VllmConfig(model=str(T_MODEL), port=8010, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768, dtype='bfloat16',
                  gpu_memory_utilization=0.80, enable_auto_tool_choice=True, enable_lora=False, tensor_parallel_size=4, startup_timeout=60 * 15)
TCMD = list(VllmServer(tcfg, adapter_manifest=None).build_cmd()) + ['--max-logprobs', '20', '--max-num-batched-tokens', '2048', '--max-model-len', '33000']
RESULTS['teacher_cmd'] = ' '.join(map(str, TCMD)); save_results(); print('teacher cmd:', RESULTS['teacher_cmd'], flush=True)
TURL = 'http://127.0.0.1:8010'
tlog = open(G2 / 'teacher_vllm.log', 'w')
tproc = subprocess.Popen([str(c) for c in TCMD], stdout=tlog, stderr=subprocess.STDOUT)
t_boot = time.time(); ready = False
while time.time() - t_boot < 15 * 60 and tproc.poll() is None:
    try:
        if requests.get(TURL + '/health', timeout=5).status_code == 200: ready = True; break
    except Exception:
        pass
    time.sleep(10)
RESULTS['teacher_boot_s'] = round(time.time() - t_boot); RESULTS['teacher_ready'] = ready; save_results()
STORE = O.Store(K)
def _save():
    STORE.save(G2 / 'teacher_scores.npz', G2 / 'teacher_meta.json'); RESULTS['teacher'] = O.summary(STORE); save_results()
if not ready:
    RESULTS['teacher_error'] = 'teacher vLLM did not become ready: ' + (G2 / 'teacher_vllm.log').read_text(errors='ignore')[-3000:]; save_results(); print(RESULTS['teacher_error'])
elif LEAVES:
    TMODEL = requests.get(TURL + '/v1/models', timeout=30).json()['data'][0]['id']
    def job(leaf):
        t = time.time()
        act, ti, tl = O.parse_prompt_logprobs(O.score_leaf(TURL, TMODEL, leaf['ids'], K), leaf['ids'], K)
        return act, ti, tl, time.time() - t
    try:   # probe on the shortest leaf: shape and speed before the full pass
        a, ti, tl, sec = job(LEAVES[0]); STORE.add(LEAVES[0], a, ti, tl, sec); _save()
        RESULTS['teacher_probe'] = {'tokens': len(LEAVES[0]['ids']), 'seconds': round(sec, 1), 'tok_per_s': round(len(LEAVES[0]['ids']) / max(sec, 1e-6)), 'act_lp_head': [float(x) for x in a[1:6]],
                                    'top_ids_head': ti[1].tolist()}
        print('teacher probe:', RESULTS['teacher_probe'], flush=True)
        n_ok, errs = O.score_all(LEAVES[1:], job, STORE, lambda: (time.time() - T_START) / 60 > BUDGET_MIN - 8, _save, workers=2, every=10)
        RESULTS['teacher_pass'] = {'scored': n_ok + 1, 'of': len(LEAVES), 'errors': len(errs), 'first_errors': errs[:3], 'minutes': round((time.time() - T_START) / 60, 1)}
    except Exception:
        RESULTS['teacher_error'] = traceback.format_exc()[-2500:]
    _save(); print('teacher pass:', RESULTS.get('teacher_pass'), RESULTS.get('teacher_error', ''), '| summary:', RESULTS.get('teacher'), flush=True)
tproc.terminate()
print('\n===== G20 SUMMARY =====')
for tag in ['smoke'] + [f'e1_{k}' for k in range(6)]:
    if tag in RESULTS and 'n' in RESULTS[tag]: print(tag, {kk: v for kk, v in RESULTS[tag].items() if kk not in ('resolved_ids', 'proxy')})
print('student total:', RESULTS.get('e1_total'), '| leaves:', RESULTS.get('leaves'), '| teacher:', RESULTS.get('teacher_pass'), RESULTS.get('teacher'))
print('capture MB:', round((G2 / 'capture.jsonl.gz').stat().st_size / 1e6, 1))
RESULTS['finished'] = time.strftime('%Y-%m-%d %H:%M:%S'); save_results()
_wd.cancel()